# Tutorial 150 — Create a custom QAChat tool: a research briefing

Most questions in QAChat are answered by retrieval plus the answering model.
Some requests need more than that: an exact count, every mention of a term,
or a structured report that must follow a fixed format. QAChat handles these
with **tools**: Python modules that perform one explicit operation for an
intent.

This tutorial builds on the tool QAChat ships as its reference example,
`generate_paper_summary`. It turns one paper into a one-page summary in the
spirit of Nature's *Research Briefing* (a short overview that unpacks a paper
for a broad scientific audience), checks the structure in Python, repairs it
once if needed, and saves Markdown in the project's `target/` folder.

Why a tool and not just a prompt? A tool is repeatable, has a defined input
and output, can be tested without a model, and can **check** its result
instead of trusting that the model followed the instructions.

The pieces, from configuration to code:

```
question → router → intent (project_types/literature.json)
                       ↓ names a tool
              tool registry (qachat/tools/registry.py)
                       ↓
     tool module (qachat/tools/generate_paper_summary.py)
          uses: load_target, take_notes, report templates
                       ↓
            answer + evidence + saved file in target/
```

## 1. Prepare QAChat

Each tutorial is independently runnable. These cells locate the repository,
make the local package importable, and call `prepare_notebook()` to verify the
project `.venv` environment. If Jupyter asks you to select the printed `.venv`
kernel, select it and rerun the cell.

In [ ]:
from pathlib import Path

root = Path.cwd()
if not (root / "qachat").is_dir():
    root = root.parent

print(f"QAChat root: {root}")

In [ ]:
import sys

if str(root) not in sys.path:
    sys.path.insert(0, str(root))

In [ ]:
from setup.notebook import prepare_notebook

root = prepare_notebook(root)

In [ ]:
import json

import qachat
from IPython.display import Markdown, display

## 2. Where a tool is switched on: the project type

A project type is one JSON file in `qachat/project_types/`. Each intent
describes what the user wants; the router reads the descriptions. An intent
can name a **tool** and its settings. The router never sees Python: it only
chooses the intent, and the configuration decides what runs.

In [ ]:
types_folder = root / "qachat" / "project_types"
literature = json.loads(
    (types_folder / "literature.json").read_text(encoding="utf-8")
)
print(json.dumps(literature["intents"]["paper_summary"], indent=2))

The registry is the controlled boundary between that configuration and code.
A project type can only name tools that are registered, and each tool
validates its own settings when the project type is loaded.

In [ ]:
from qachat.tools import TOOL_REGISTRY

for name, tool in TOOL_REGISTRY.items():
    print(f"{name:24} {tool.summary}")

## 3. The contract: what a tool receives and returns

Every tool has the same shape:

- a **settings** dataclass with `parse_settings(data, where)`;
- `run(settings, request, context) -> ToolResult`.

`ToolRequest` is what the user asked. `ToolContext` carries QAChat's
services as plain functions (indexed chunks, ranking, the models, progress,
confirmation, tracing), so a tool never imports the RAG session and can be
tested with simple stand-ins. `ToolResult` is the answer, the evidence it
cites, and a summary saved with the turn.

In [ ]:
from dataclasses import fields

from qachat.tools import ToolContext, ToolRequest, ToolResult

for contract in (ToolRequest, ToolContext, ToolResult):
    names = ", ".join(field.name for field in fields(contract))
    print(f"{contract.__name__}: {names}")

## 4. The template: structure as data

Report formats live in `qachat/report_templates/` as JSON, not in Python.
QAChat ships two: `research_summary` (ten compact sections) and
`research_briefing`, the one-page *Research Briefing* format of
Nature-family journals: a headline of at most 90 characters, then six
sections with word limits taken from Nature's author template, and at most
750 words in total so it fits on one page.

A section can also say what it is written **from**: `beginning` (the
paper's first page, for the citation) or `references` (its reference list,
so suggested reading comes from papers the authors actually cite).

In [ ]:
from qachat.report_templates import load_template

template = load_template("research_briefing")
print(template.reference, "\n")
print(f"Headline: at most {template.headline_characters} characters")
print(f"Total: at most {template.max_total_words} words\n")
for section in template.sections:
    limit = section.words or section.items
    print(f"- {section.name} {limit} from {section.source}")

## 5. Checking the structure in Python

Asking a model to "follow this format exactly" is not a guarantee.
`check_report` verifies the result: every required section present, not
repeated, not empty, in order, within its word or item limit, and a
headline within its length. A report built to fit passes.

In [ ]:
from qachat.report_templates import check_report


def filler(section):
    """Content that fits one section's limit."""

    if section.items:
        return "- First paper.\n- Second paper."
    low, high = section.words
    return " ".join(["word"] * ((low + high) // 2))


fitting = "\n".join(
    ["# Plants reveal stress before it shows"]
    + [f"## {s.name}\n{filler(s)}" for s in template.sections]
)
print("valid:", check_report(fitting, template).valid)

An incomplete or overlong report fails, and the check says exactly what is
wrong. The tool uses these sentences to ask the model for a targeted fix.

In [ ]:
broken = "# " + "x" * 100 + "\n## The Reason\nToo short."
for problem in check_report(broken, template).problems():
    print("-", problem)

## 6. The target folder

A project has two kinds of documents. `sources/` is the knowledge base: it is
indexed and searched. `target/` holds what you are working on, such as a
paper to summarize and the summaries QAChat writes. It is **never indexed**,
so a generated report can never come back as a source.

This tutorial uses a small project folder with one paper in `target/`. The
tool can also summarize any paper among an indexed project's sources.

In [ ]:
import shutil

from qachat.tools.load_target import load_target, target_folder

project_root = root / "sandbox" / "tutorial_150_custom_tool"
paper = root / "example_repos" / "literature_repo" / "papers"
paper = paper / "patil25deep.pdf"

target = target_folder(project_root)
shutil.copy2(paper, target / paper.name)

document = load_target(project_root, paper.name)
print(f"{document.path}: {len(document.passages)} passages, "
      f"about {document.estimated_tokens:,} tokens")

## 7. Run the tool with stand-in services

Before using a real model, run the tool with scripted replies. This is how
QAChat's tests work. The tool writes the headline and **each section in
its own call**, and Python assembles the headings, so a section can never
go missing or out of order. Each piece is checked as it arrives.

Here the scripted model first writes a placeholder citation. The source
check sees that it does not match the paper's first page, and the tool
asks once for a fix; the second reply copies the real title.

In [ ]:
import re

from qachat.trace_logging import NullTraceLogger
from qachat.tools import ToolUse, run_tool
from qachat.tools.generate_paper_summary import (
    SummarySettings,
    paper_beginning,
)

first_line = paper_beginning(document.passages).splitlines()[0]
steps = []


def scripted_model(system, prompt, max_new_tokens):
    if "take notes" in system:
        return "- The method is called DIVA [1]"
    section = re.search(r'"(.+?)" section', prompt).group(1)
    if section == "Citation":
        if "revise" in system:
            return first_line + " [1]"
        return "Author, A. et al. Title. Journal (year)."
    return "Summarized from the notes [1]."


def make_context(model):
    return ToolContext(
        project_root=project_root,
        chunks=[],
        graph=qachat.CodeGraph(entities=(), relations=()),
        rank=lambda text: [],
        retrieval_count=20,
        unsearchable=lambda indexed: [],
        source_files=lambda: {},
        ask_router=model,
        ask_model=model,
        write=model,
        model_id="scripted",
        progress=lambda message, **details: steps.append(message),
        done=lambda message, **details: steps.append(message),
        confirm=None,
        trace=NullTraceLogger(),
        trace_turn_id=None,
    )

In [ ]:
use = ToolUse("generate_paper_summary", SummarySettings())
request = ToolRequest(
    question="turn my paper in target into a research summary",
    text="turn my paper in target into a research summary",
)
result = run_tool(use, request, make_context(scripted_model))

for step in steps:
    print("·", step)
print()
print("fixes:", result.summary["repairs"])
print("valid:", result.summary["valid"])
print("saved:", result.summary["saved"])

## 8. Run it with the real model

Now the same tool with QAChat's default chat model and the research
briefing template. It reads the paper in parts (a whole paper is too long
for one prompt on a small model), writes the headline and each section from
the notes, checks and fixes each piece, and saves the briefing. On an Apple
M5 with the 8B model this takes about four minutes for a 30-page paper.

In [ ]:
from qachat.answering import side_reply

backend = qachat.choose_backend(qachat.BACKEND_AUTO)
model_id = (
    qachat.DEFAULT_CHAT_MODEL_MLX
    if backend == "mlx"
    else qachat.DEFAULT_CHAT_MODEL_TORCH
)
chat = qachat.ChatSession(
    model_id,
    backend=backend,
    thinking=False,
    temperature=0.0,
    allow_download=True,
)


def model(system, prompt, max_new_tokens):
    return side_reply(chat, system, prompt, max_new_tokens=max_new_tokens)

In [ ]:
steps.clear()
briefing = ToolUse(
    "generate_paper_summary",
    SummarySettings(template="research_briefing", repair_attempts=2),
)
request = ToolRequest(
    question="turn my paper in target into a research briefing",
    text="turn my paper in target into a research briefing",
)
result = run_tool(briefing, request, make_context(model))
print("\n".join(f"· {step}" for step in steps))

In [ ]:
print("valid:", result.summary["valid"], result.summary["problems"])
saved = project_root / result.summary["saved"]
display(Markdown(saved.read_text(encoding="utf-8")))

In the desktop app, the same tool runs when you ask a literature project
something like *"Turn the DIVA paper into a research briefing"*: the router
picks the `paper_summary` intent, and the answer shows the summary with
clickable passage citations and where the file was saved.

## 9. Write your own tool

A new tool needs one module, one registry entry, and an intent that names
it. Below is a complete, minimal tool that needs no model: it counts the
words of a target file. Copy this shape for your own tools.

In [ ]:
from dataclasses import dataclass

from qachat.tools.contract import check_int, reject_unknown


@dataclass(frozen=True, slots=True)
class WordCountSettings:
    """Settings of the example ``word_count`` tool."""

    top_words: int = 5


def parse_settings(data: dict, where: str) -> WordCountSettings:
    """Validate the tool's settings from a project type."""

    reject_unknown(where, data, ("top_words",))
    return WordCountSettings(
        top_words=check_int(where, data, "top_words", 5, 1, 50)
    )


def run(settings, request, context) -> ToolResult:
    """Count the words of the target file the question is about."""

    document = load_target(context.project_root, paper.name)
    words = " ".join(p.text for p in document.passages).split()
    context.done(f"Counted {len(words)} words")
    answer = f"{document.path} has about {len(words):,} words."
    return ToolResult(answer, (), {"name": "word_count"})

In [ ]:
settings = parse_settings({"top_words": 3}, "example.intents.count.tool")
print(run(settings, request, make_context(scripted_model)).answer)

To make it available in QAChat:

1. Save the module as `qachat/tools/word_count.py`.
2. Register it in `qachat/tools/registry.py`: add `"word_count"` to
   `ToolName`, a `TOOL_WORD_COUNT` constant, and a `Tool(...)` entry with its
   `parse_settings` and `run`.
3. Name it in an intent of `qachat/project_types/literature.json`:

```json
"word_count": {
  "description": "Count the words of a paper in the target folder.",
  "history": {"policy": "none"},
  "retrieval": {"mode": "none"},
  "tool": {"name": "word_count", "top_words": 3}
}
```

Registration makes a tool *available*; the project type decides *when* it is
used. Ideas that fit the same pattern: comparing two papers, extracting
methods or statistics, screening papers against criteria, or checking a
manuscript against a reporting guideline.